In [1]:
import torch 
import numpy as np
import matplotlib.pyplot as plt
import sys, pathlib

SRC = pathlib.Path.cwd().parent / "moe" / "src"   # ...\LearningDeepLearning\moe\src
sys.path.insert(0, str(SRC))

from gmm_dataset import generate_dataset

In [2]:
class SimpleDNN(torch.nn.Module):
    def __init__(self, D, H, N):
        super(SimpleDNN, self).__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(D, H),
            torch.nn.ReLU(),
            torch.nn.LazyBatchNorm1d(),
            torch.nn.Dropout(),
            torch.nn.Linear(H, H),
            torch.nn.ReLU(),
            torch.nn.LazyBatchNorm1d(),
            torch.nn.Dropout(),
            torch.nn.Linear(H, H),
            torch.nn.ReLU(),
            torch.nn.Dropout(),
            torch.nn.LazyBatchNorm1d(),
            torch.nn.Linear(H, N)
        )

    def forward(self, x):
        return self.net.forward(x)

def fit_batch(moe, train_dataloader, test_dataloader, criterion, D_out, num_epochs=2):
    optim = torch.optim.Adam(moe.parameters(), lr=1e-3)

    losses = []
    
    for i in range(num_epochs):
        moe.train()
        for (X, target_tensor) in train_dataloader:
            moe.to(X)
            optim.zero_grad()
            pred = moe.forward(X)
            loss = criterion(pred, target_tensor)
            loss.backward()
            optim.step()

            losses.append(loss.item())

        moe.eval()
        correct = 0 
        total = 0
        for (X_test, Y_test) in test_dataloader:
            preds = moe.forward(X_test)
            assert preds.shape == (X_test.shape[0], D_out), f"shape of preds {preds.shape}, is not {X_test} * {D_out}"
            preds = preds.argmax(dim=-1)
            batch_correct = (preds == Y_test).sum()
            correct += batch_correct
            total += X_test.shape[0]

        acc = correct / total 
        print(f"Epoch {i} acc {acc}")


    plt.scatter(range(len(losses)), losses)
    plt.savefig('loss_plot.png') 
    plt.close()

In [3]:
M = 100000
D = 2
N = 4
H = 1000
batch_size = 1000
criterion = torch.nn.CrossEntropyLoss()
train_dataloader, test_dataloader = generate_dataset(M=M, D=D, N=N, batch_size=batch_size)
model = SimpleDNN(D, H, N)
fit_batch(model, train_dataloader, test_dataloader, criterion, N, num_epochs=20)




/home/ajrfhp/LearningDeepLearning/moe/src/gmm_dataset.py:43: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


Epoch 0 acc 0.49978750944137573
Epoch 1 acc 0.8772249817848206
Epoch 2 acc 0.9405999779701233
Epoch 3 acc 0.9383500218391418
Epoch 4 acc 0.9403125047683716
Epoch 5 acc 0.9448124766349792
Epoch 6 acc 0.9329749941825867
Epoch 7 acc 0.9495624899864197
Epoch 8 acc 0.9466875195503235
Epoch 9 acc 0.9469749927520752
Epoch 10 acc 0.9423750042915344
Epoch 11 acc 0.9437749981880188
Epoch 12 acc 0.9416624903678894
Epoch 13 acc 0.9424499869346619
Epoch 14 acc 0.9392750263214111
Epoch 15 acc 0.9482125043869019
Epoch 16 acc 0.9460874795913696
Epoch 17 acc 0.9451375007629395
Epoch 18 acc 0.9519249796867371
Epoch 19 acc 0.9504374861717224
